# 🏦 Банковский счёт: `Account` и `CreditAccount`

> **Цель:** реализовать класс `Account`, моделирующий банковский счёт, с историей операций и аналитикой.  
> Затем создать наследника `CreditAccount`.

---

## 📌 1. Класс `Account`

### Инициализация

```python
Account(account_holder, balance=0)
```

| Параметр | Тип | По умолчанию | Описание |
|----------|-----|--------------|----------|
| `account_holder` | `str` | — | Имя владельца счёта |
| `balance` | `float` | `0` | Начальный баланс. Не может быть отрицательным |

### Атрибуты

| Атрибут | Тип | Описание |
|---------|-----|----------|
| `holder` | `str` | Имя владельца |
| `_balance` | `float` | Приватный текущий баланс |
| `operations_history` | `list` / `tuple` | История операций |

> ⚠️ **Важно:** каждая операция хранится как структурированная запись — словарь или кортеж.  
> Минимальный набор полей:
> - тип операции: `'deposit'` или `'withdraw'`;
> - сумма;
> - дата и время;
> - текущий баланс после операции;
> - статус: `'success'` или `'fail'`.

#### Пример записи операции

```python
{
    "type": "deposit",
    "amount": 1000.0,
    "datetime": datetime.now(),
    "balance_after": 1000.0,
    "status": "success"
}
```

---

## 🛠️ 2. Методы `Account`

### `__init__(self, account_holder, balance=0)`
Конструктор класса.

### `deposit(self, amount)`
Пополнение счёта.

- принимает сумму;
- сумма должна быть положительной;
- при успехе обновляет баланс и добавляет запись в историю.

### `withdraw(self, amount)`
Снятие средств.

- принимает сумму;
- сумма должна быть положительной;
- проверяет, достаточно ли средств;
- если средств недостаточно — операция не проходит, но попытка фиксируется в истории со статусом `'fail'`;
- при успехе обновляет баланс и добавляет запись.

### `get_balance(self)`
Возвращает текущий баланс.

### `get_history(self)`
Возвращает историю операций.

> 💡 **Подсказка:** для работы с датой и временем используйте `datetime.now()` из модуля `datetime`.

---

## 💳 3. Класс `CreditAccount(Account)`

Наследует всю функциональность `Account` и добавляет кредитную логику.

### Особенности

- При инициализации принимает дополнительный параметр `credit_limit`.
- Баланс может быть отрицательным, но не ниже `-credit_limit`.
- По запросу показывает доступные кредитные средства:  
  `доступно = текущий баланс + credit_limit`
- В историю операций добавляется информация о том, были ли использованы кредитные средства.

### Инициализация

```python
CreditAccount(account_holder, balance=0, credit_limit=0)
```

---

## ✅ Чек-лист реализации

- [ ] Создан класс `Account`.
- [ ] Реализован конструктор с проверкой баланса.
- [ ] Реализованы `deposit`, `withdraw`, `get_balance`, `get_history`.
- [ ] История операций хранит структурированные записи.
- [ ] Неудачные попытки снятия фиксируются.
- [ ] Создан класс `CreditAccount`.
- [ ] Учтён кредитный лимит.
- [ ] Добавлена информация об использовании кредитных средств.

---

## 📚 Полезные импорты

```python
from datetime import datetime
```

---

> **Совет:** для красивого вывода истории можно вернуть список словарей или преобразовать его в `pandas.DataFrame`.

# 📊 Критерии оценивания проекта

> **Максимум по базовым критериям: 8 баллов** — это оценка **«отлично»**.  
> 💡 Преподаватель может добавить **1–2 балла** и поставить **9 или 10**, если работа выполнена сверх ожиданий и выходит за рамки программы.

---

| № | Критерий | Баллы | Описание |
|:--:|:---------|:-----:|:---------|
| 1 | 🧱 **Корректность ООП** | **3** | Класс корректно инициализируется, используются инкапсуляция (приватные атрибуты, геттеры), методы `deposit` и `withdraw` работают без ошибок, логически верно обрабатывают неверные входные данные (отрицательные суммы) |
| 2 | 📜 **История операций** | **2** | История сохраняется в структурированном виде. Метод `get_history` возвращает информацию в удобном формате |
| 3 | 🕒 **Работа с датами** | **1** | В каждой операции корректно фиксируется дата и время её проведения с помощью модуля `datetime` |
| 4 | 🧬 **Наследование** | **2** | Корректно реализована функциональность класса `CreditAccount(Account)`: в реализации учтены все заданные особенности кредитного счёта |
| 5 | ✨ **Чистота кода** | **−1** за каждое нарушение | Код откомментирован, читаемый, соответствует PEP8 |

---

> **Итого:** `3 + 2 + 1 + 2 = 8` баллов — оценка **«отлично»**.  
> 🚀 Дополнительные баллы: **9–10**, если работа выходит за рамки программы.

In [ ]:
from dataclasses import dataclass
from datetime import datetime, timezone
from decimal import Decimal, ROUND_HALF_EVEN
from enum import StrEnum
from uuid import UUID, uuid4


ZERO_MONEY = Decimal("0.00")
MONEY_QUANTUM = Decimal("0.01")


def normalize_money(value: object) -> Decimal:
    """Проверяет и округляет денежное значение до двух знаков."""

    if not isinstance(value, Decimal):
        raise TypeError("Денежное значение должно иметь тип Decimal")

    if not value.is_finite():
        raise ValueError("Денежное значение должно быть конечным числом")

    return value.quantize(
        MONEY_QUANTUM,
        rounding=ROUND_HALF_EVEN,
    )


class OperationType(StrEnum):
    """Определяет тип банковской операции."""

    DEPOSIT = "deposit"
    WITHDRAW = "withdraw"


class OperationStatus(StrEnum):
    """Определяет результат банковской операции."""

    SUCCESS = "success"
    FAIL = "fail"


class FailureReason(StrEnum):
    """Определяет причину отказа в операции."""

    INSUFFICIENT_FUNDS = "insufficient_funds"
    CREDIT_LIMIT_EXCEEDED = "credit_limit_exceeded"
    ACCOUNT_BLOCKED = "account_blocked"


class Currency(StrEnum):
    """Определяет валюту банковского счёта."""

    RUB = "RUB"
    USD = "USD"
    EUR = "EUR"


class AccountStatus(StrEnum):
    """Определяет состояние банковского счёта."""

    ACTIVE = "active"
    BLOCKED = "blocked"


class ActorType(StrEnum):
    """Определяет тип инициатора события."""

    ACCOUNT_HOLDER = "account_holder"
    BANK_EMPLOYEE = "bank_employee"
    SYSTEM = "system"


class AccountEventType(StrEnum):
    """Определяет тип события банковского счёта."""

    BLOCKED = "blocked"
    UNBLOCKED = "unblocked"


class BlockReason(StrEnum):
    """Определяет причину блокировки счёта."""

    USER_REQUEST = "user_request"
    FRAUD_SUSPICION = "fraud_suspicion"
    BANK_DECISION = "bank_decision"
    LEGAL_RESTRICTION = "legal_restriction"


@dataclass(frozen=True, slots=True)
class EventActor:
    """Представляет инициатора события счёта."""

    actor_type: ActorType
    actor_id: UUID | None = None

    def __post_init__(self) -> None:
        """Проверяет согласованность типа и ID инициатора."""

        if self.actor_type is ActorType.SYSTEM and self.actor_id is not None:
            raise ValueError("Системный инициатор не должен иметь actor_id")

        if self.actor_type is not ActorType.SYSTEM and self.actor_id is None:
            raise ValueError("Владелец или сотрудник должен иметь actor_id")


@dataclass(frozen=True, slots=True)
class AccountEvent:
    """Представляет неизменяемое событие счёта."""

    event_id: UUID
    account_id: UUID
    event_type: AccountEventType
    created_at: datetime
    actor: EventActor
    block_reason: BlockReason | None = None

    def __post_init__(self) -> None:
        """Проверяет согласованность статуса блокировки и причины."""

        if self.event_type is AccountEventType.BLOCKED and self.block_reason is None:
            raise ValueError("Событие блокировки должно иметь причину")

        if (
            self.event_type is AccountEventType.UNBLOCKED
            and self.block_reason is not None
        ):
            raise ValueError(
                "Событие разблокировки не может содержать причину блокировки"
            )


@dataclass(frozen=True, slots=True)
class Transaction:
    """Представляет неизменяемую банковскую операцию."""

    transaction_id: UUID
    account_id: UUID
    currency: Currency
    operation_type: OperationType
    amount: Decimal
    created_at: datetime
    balance_after: Decimal
    status: OperationStatus
    failure_reason: FailureReason | None = None
    credit_used: bool | None = None

    def __post_init__(self) -> None:
        """Проверяет согласованность статуса и причины отказа."""

        if self.status is OperationStatus.SUCCESS and self.failure_reason is not None:
            raise ValueError("Успешная операция не может иметь причину отказа")

        if self.status is OperationStatus.FAIL and self.failure_reason is None:
            raise ValueError("Неуспешная операция должна иметь причину отказа")

    @property
    def is_successful(self) -> bool:
        """Показывает, была ли операция успешной."""

        return self.status is OperationStatus.SUCCESS


class Account:
    """Моделирует банковский счёт и хранит историю операций."""

    def __init__(
        self,
        account_holder: str,
        balance: Decimal = ZERO_MONEY,
        currency: Currency = Currency.RUB,
        account_id: UUID | None = None,
        account_status: AccountStatus = AccountStatus.ACTIVE,
    ) -> None:
        """Создаёт банковский счёт с указанным владельцем и начальным балансом."""

        balance = normalize_money(balance)

        if balance < ZERO_MONEY:
            raise ValueError("Начальный баланс не может быть отрицательным")

        if account_id is not None and not isinstance(account_id, UUID):
            raise TypeError("Идентификатор счёта должен иметь тип UUID")

        if not isinstance(currency, Currency):
            raise TypeError("Валюта должна иметь тип Currency")

        if not isinstance(account_status, AccountStatus):
            raise TypeError("Статус счета должен иметь тип AccountStatus")

        self._currency: Currency = currency
        self._account_id: UUID = uuid4() if account_id is None else account_id
        self.holder: str = account_holder
        self._balance: Decimal = balance
        self._account_status: AccountStatus = account_status
        self._block_reason: BlockReason | None = None

        self._operations_history: list[Transaction] = []
        self._events_history: list[AccountEvent] = []

    def get_account_status(self) -> AccountStatus:
        """Возвращает статус банковского счета."""

        return self._account_status

    def get_block_reason(self) -> BlockReason | None:
        """Возвращает текущую причину блокировки счёта."""

        return self._block_reason

    def block_account(
        self, block_reason: BlockReason, actor: EventActor
    ) -> AccountEvent | None:
        """Блокирует счёт по указанной причине."""

        if not isinstance(block_reason, BlockReason):
            raise TypeError("Причина блокировки должна иметь тип BlockReason")

        if not isinstance(actor, EventActor):
            raise TypeError("Инициатор должен иметь тип EventActor")

        if self.is_blocked:
            return None

        self._account_status = AccountStatus.BLOCKED
        self._block_reason = block_reason

        return self._add_account_event(
            event_type=AccountEventType.BLOCKED,
            block_reason=block_reason,
            actor_event=actor,
        )

    def unblock_account(self, actor: EventActor) -> AccountEvent | None:
        """Разблокирует счёт."""

        if not isinstance(actor, EventActor):
            raise TypeError("Инициатор должен иметь тип EventActor")

        if not self.is_blocked:
            return None

        self._account_status = AccountStatus.ACTIVE
        self._block_reason = None

        return self._add_account_event(
            event_type=AccountEventType.UNBLOCKED, actor_event=actor
        )

    def _get_available_withdrawal_funds(self) -> Decimal:
        """Возвращает сумму, доступную для снятия."""

        return self._balance

    @property
    def is_blocked(self) -> bool:
        """Показывает, заблокирован ли счёт."""

        return self._account_status is AccountStatus.BLOCKED

    def get_currency(self) -> Currency:
        """Возвращает валюту счёта."""

        return self._currency

    def get_account_id(self) -> UUID:
        """Возвращает идентификатор счёта."""

        return self._account_id

    def _get_insufficient_funds_reason(self) -> FailureReason:
        """Возвращает причину отказа при нехватке средств."""

        return FailureReason.INSUFFICIENT_FUNDS

    def _get_credit_used(self, status: OperationStatus) -> bool | None:
        """Возвращает признак использования кредита."""

        return None

    def deposit(self, amount: Decimal) -> Transaction:
        """Пополняет счёт на положительную сумму."""

        amount = normalize_money(amount)

        if amount <= ZERO_MONEY:
            raise ValueError("Сумма пополнения должна быть положительной")

        self._balance += amount

        return self._add_history(
            operation_type=OperationType.DEPOSIT,
            amount=amount,
            date_time=datetime.now(timezone.utc),
            balance_after=self._balance,
            status=OperationStatus.SUCCESS,
        )

    def withdraw(self, amount: Decimal) -> Transaction:
        """Снимает средства в пределах доступной суммы."""

        amount = normalize_money(amount)

        if amount <= ZERO_MONEY:
            raise ValueError("Сумма снятия должна быть положительной")

        if self.is_blocked:
            status = OperationStatus.FAIL
            failure_reason = FailureReason.ACCOUNT_BLOCKED
        elif amount > self._get_available_withdrawal_funds():
            status = OperationStatus.FAIL
            failure_reason = self._get_insufficient_funds_reason()
        else:
            self._balance -= amount
            status = OperationStatus.SUCCESS
            failure_reason = None

        credit_used = self._get_credit_used(status)

        return self._add_history(
            operation_type=OperationType.WITHDRAW,
            amount=amount,
            date_time=datetime.now(timezone.utc),
            balance_after=self._balance,
            status=status,
            failure_reason=failure_reason,
            credit_used=credit_used,
        )

    def get_balance(self) -> Decimal:
        """Возвращает текущий баланс счёта."""

        return self._balance

    def _add_history(
        self,
        operation_type: OperationType,
        amount: Decimal,
        date_time: datetime,
        balance_after: Decimal,
        status: OperationStatus,
        failure_reason: FailureReason | None = None,
        credit_used: bool | None = None,
    ) -> Transaction:
        """Добавляет запись об операции в историю счёта."""

        transaction = Transaction(
            transaction_id=uuid4(),
            account_id=self._account_id,
            currency=self._currency,
            operation_type=operation_type,
            amount=amount,
            created_at=date_time,
            balance_after=balance_after,
            status=status,
            failure_reason=failure_reason,
            credit_used=credit_used,
        )

        self._operations_history.append(transaction)
        return transaction

    def _add_account_event(
        self,
        event_type: AccountEventType,
        actor_event: EventActor,
        block_reason: BlockReason | None = None,
    ) -> AccountEvent:
        """Добавляет событие в историю счёта."""

        event = AccountEvent(
            event_id=uuid4(),
            account_id=self._account_id,
            event_type=event_type,
            created_at=datetime.now(timezone.utc),
            actor=actor_event,
            block_reason=block_reason,
        )
        self._events_history.append(event)
        return event

    def get_history(self) -> tuple[Transaction, ...]:
        """Возвращает неизменяемую историю операций."""

        return tuple(self._operations_history)

    def get_events_history(self) -> tuple[AccountEvent, ...]:
        """Возвращает неизменяемую историю событий счёта."""

        return tuple(self._events_history)

    def __str__(self) -> str:
        """Возвращает строковое представление банковского счёта."""

        return (
            f"Владелец: {self.holder}. Баланс: {self._balance} {self._currency.value}"
        )


class CreditAccount(Account):
    """Моделирует банковский счёт с кредитным лимитом."""

    def __init__(
        self,
        account_holder: str,
        balance: Decimal = ZERO_MONEY,
        credit_limit: Decimal = ZERO_MONEY,
        currency: Currency = Currency.RUB,
        account_id: UUID | None = None,
        account_status: AccountStatus = AccountStatus.ACTIVE,
    ) -> None:
        """Создаёт банковский счёт с балансом и кредитным лимитом."""

        super().__init__(
            account_holder=account_holder,
            balance=balance,
            account_id=account_id,
            currency=currency,
            account_status=account_status,
        )

        credit_limit = normalize_money(credit_limit)

        if credit_limit < ZERO_MONEY:
            raise ValueError("Кредитный лимит не может быть отрицательным")

        self._credit_limit: Decimal = credit_limit

    def _get_available_withdrawal_funds(self) -> Decimal:
        """Возвращает средства с учётом кредитного лимита."""

        return self.get_available_funds()

    def _get_insufficient_funds_reason(self) -> FailureReason:
        """Возвращает причину превышения кредитного лимита."""

        return FailureReason.CREDIT_LIMIT_EXCEEDED

    def _get_credit_used(self, status: OperationStatus) -> bool | None:
        """Показывает, использованы ли кредитные средства."""

        return status is OperationStatus.SUCCESS and self._balance < ZERO_MONEY

    def get_available_funds(self) -> Decimal:
        """Возвращает общую сумму доступных собственных и кредитных средств."""

        return self._balance + self._credit_limit

    def get_credit_limit(self) -> Decimal:
        """Возвращает кредитный лимит."""

        return self._credit_limit

    def __str__(self) -> str:
        """Возвращает строковое представление кредитного счёта."""

        available_funds = self.get_available_funds()

        return (
            f"Владелец: {self.holder}. "
            f"Баланс: {self._balance} {self._currency.value}. "
            f"Кредитный лимит: {self._credit_limit} {self._currency.value}. "
            f"Доступно средств: {available_funds} {self._currency.value}."
        )